In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle('./FINAL_MAPPING_1789_2023.pkl')

In [3]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm.notebook import tqdm
import numpy as np

def precompute_and_add_embeddings(df, description_column='Description_N', 
                                 model_name='sentence-transformers/all-mpnet-base-v2',
                                 batch_size=32,
                                 embedding_column='description_embedding',
                                 token=None):
    """
    Precompute embeddings for a description column in a DataFrame and add them as a new column.
    
    Parameters:
    - df: DataFrame containing the descriptions
    - description_column: Name of the column containing text descriptions
    - model_name: Name of the transformer model to use
    - batch_size: Number of descriptions to process at once
    - embedding_column: Name of the new column to store embeddings
    - token: Optional HuggingFace token
    
    Returns:
    - DataFrame with added embeddings column
    """
    # Initialize model and tokenizer
    print("Loading model and tokenizer...")
    if token:
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
        model = AutoModel.from_pretrained(model_name, token=token)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    model = model.to(device)
    model.eval()
    
    # Create a copy of the DataFrame to avoid modifying the original
    result_df = df.copy()
    
    # Prepare to store embeddings
    all_embeddings = []
    
    # Get descriptions and handle missing values
    descriptions = df[description_column].fillna("").astype(str).tolist()
    
    # Process in batches with progress tracking
    total_batches = (len(descriptions) + batch_size - 1) // batch_size
    
    with tqdm(total=len(descriptions), desc="Computing embeddings") as pbar:
        for i in range(0, len(descriptions), batch_size):
            # Get batch and handle end of list
            end_idx = min(i + batch_size, len(descriptions))
            batch_descriptions = descriptions[i:end_idx]
            
            # Skip empty batch (shouldn't happen but just in case)
            if not batch_descriptions:
                continue
            
            # Tokenize
            inputs = tokenizer(batch_descriptions, padding=True, truncation=True, 
                             return_tensors="pt").to(device)
            
            # Generate embeddings
            with torch.no_grad():
                outputs = model(**inputs)
                # Mean pooling - taking mean of all token embeddings
                attention_mask = inputs['attention_mask'].unsqueeze(-1)
                token_embeddings = outputs.last_hidden_state
                input_mask_expanded = attention_mask.expand(token_embeddings.size()).float()
                batch_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
                # Move to CPU and convert to numpy for storage
                batch_embeddings = batch_embeddings.detach().cpu().numpy()
            
            # Store embeddings
            all_embeddings.extend(batch_embeddings)
            
            # Update progress bar
            pbar.update(len(batch_descriptions))
    
    print(f"Generated {len(all_embeddings)} embeddings")
    
    # Add embeddings to DataFrame
    # Convert embeddings to list form for storage in DataFrame
    embedding_list = [embedding.tolist() for embedding in all_embeddings]
    result_df[embedding_column] = embedding_list
    
    print(f"Added embeddings as column '{embedding_column}'")
    
    return result_df

def find_similar_descriptions(query_description, df, embedding_column='description_embedding',
                             description_column='Description_N', 
                             model_name='sentence-transformers/all-mpnet-base-v2',
                             top_k=5, alpha=0.7, beta=0.2, gamma=0.1, token=None):
    """
    Find the most similar descriptions in the DataFrame based on precomputed embeddings.
    
    Parameters:
    - query_description: Description text to find matches for
    - df: DataFrame containing descriptions and their precomputed embeddings
    - embedding_column: Column name containing embeddings
    - description_column: Column name containing text descriptions
    - model_name: Model name for generating query embedding
    - top_k: Number of most similar descriptions to return
    - alpha, beta, gamma: Weights for combined similarity score
    - token: Optional HuggingFace token
    
    Returns:
    - DataFrame with top_k most similar descriptions
    """
    # Initialize model and tokenizer for the query
    if token:
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
        model = AutoModel.from_pretrained(model_name, token=token)
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    
    # Generate embedding for the query
    inputs = tokenizer(query_description, padding=True, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        attention_mask = inputs['attention_mask'].unsqueeze(-1)
        token_embeddings = outputs.last_hidden_state
        input_mask_expanded = attention_mask.expand(token_embeddings.size()).float()
        query_embedding = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        query_embedding = query_embedding.detach().cpu().numpy().flatten()
    
    # Get all embeddings from DataFrame
    all_embeddings = np.array(df[embedding_column].tolist())
    
    # Calculate cosine similarities
    # (using dot product / (norm1 * norm2) for vectorized calculation)
    norm_query = np.linalg.norm(query_embedding)
    norm_all = np.linalg.norm(all_embeddings, axis=1)
    dot_products = np.dot(all_embeddings, query_embedding)
    cosine_similarities = dot_products / (norm_query * norm_all)
    
    # Get top candidates by cosine similarity
    pre_top_k = min(top_k * 10, len(df))  # Get more candidates for reranking
    pre_top_indices = np.argsort(cosine_similarities)[-pre_top_k:][::-1]
    
    # Prepare for reranking with combined similarity
    from rapidfuzz import fuzz
    
    # Query tokens for Jaccard
    query_tokens = set(query_description.lower().split())
    
    # Calculate combined similarity for top candidates
    candidates = []
    for idx in pre_top_indices:
        description = df.iloc[idx][description_column]
        if pd.isna(description) or not description:
            continue
            
        # Get the precomputed cosine similarity
        cos_sim = cosine_similarities[idx]
        
        # Calculate Jaccard similarity
        desc_tokens = set(str(description).lower().split())
        jac_sim = len(query_tokens & desc_tokens) / len(query_tokens | desc_tokens) if desc_tokens else 0.0
        
        # Calculate Levenshtein similarity
        lev_sim = fuzz.ratio(query_description.lower(), str(description).lower()) / 100.0
        
        # Combined similarity score
        combined_sim = alpha * cos_sim + beta * jac_sim + gamma * lev_sim
        
        candidates.append((idx, cos_sim, jac_sim, lev_sim, combined_sim))
    
    # Sort by combined similarity and get top k
    candidates.sort(key=lambda x: x[4], reverse=True)
    top_indices = [x[0] for x in candidates[:top_k]]
    
    # Create result DataFrame
    result_df = df.iloc[top_indices].copy()
    
    # Add similarity scores
    for i, (idx, cos_sim, jac_sim, lev_sim, combined_sim) in enumerate(candidates[:top_k]):
        result_df.iloc[i, result_df.columns.get_loc('query_cosine_similarity')] = cos_sim
        result_df.iloc[i, result_df.columns.get_loc('query_jaccard_similarity')] = jac_sim
        result_df.iloc[i, result_df.columns.get_loc('query_levenshtein_similarity')] = lev_sim
        result_df.iloc[i, result_df.columns.get_loc('query_combined_similarity')] = combined_sim
    
    return result_df

In [4]:
# First, add necessary columns to store similarity scores
if 'query_cosine_similarity' not in df.columns:
    df['query_cosine_similarity'] = 0.0
if 'query_jaccard_similarity' not in df.columns:
    df['query_jaccard_similarity'] = 0.0
if 'query_levenshtein_similarity' not in df.columns:
    df['query_levenshtein_similarity'] = 0.0
if 'query_combined_similarity' not in df.columns:
    df['query_combined_similarity'] = 0.0

# Precompute embeddings
df_with_embeddings = precompute_and_add_embeddings(df, description_column='Description_N')

# Save the DataFrame with embeddings
df_with_embeddings.to_pickle('./df_with_embeddings.pkl')

# Later, when you need to search:
query = "Cotton cloth, bleached, containing synthetic fiber"
similar_items = find_similar_descriptions(
    query, 
    df_with_embeddings, 
    embedding_column='description_embedding',
    description_column='Description_N', 
    top_k=5
)

# Display results
for idx, row in similar_items.iterrows():
    print(f"Similarity: {row['query_combined_similarity']:.3f}")
    print(f"Description: {row['Description_N']}")
    print(f"HS Code: {row['HS_N']}")
    print("-" * 50)

Loading model and tokenizer...
Using device: cpu


Computing embeddings:   0%|          | 0/745258 [00:00<?, ?it/s]

Generated 745258 embeddings
Added embeddings as column 'description_embedding'
Similarity: 0.745
Description: Cotton cloth, bleached
HS Code: 1894_2951_cotton_cloth_bleached
--------------------------------------------------
Similarity: 0.685
Description: Cotton cloth not bleached, dyed, colored, stained, painted, or cloth printed
HS Code: 1894_2939_cotton_cloth_not_bleached_dyed_colored_s
--------------------------------------------------
Similarity: 0.668
Description: Cotton cloths
HS Code: 1828_500_cotton_cloths
--------------------------------------------------
Similarity: 0.639
Description: Bleached plain weave fabrics of cotton, under 85% cotton wt., mixed mainly with man-made fibers, n/o 200 g/m2, of number 42 or lower
HS Code: 52102140
--------------------------------------------------
Similarity: 0.639
Description: Bleached plain weave fabrics of cotton, under 85% cotton wt., mixed mainly with man-made fibers, n/o 200 g/m2, of number 42 or lower
HS Code: 52102140
-------------

In [5]:
df_with_embeddings = pd.read_pickle('./df_with_embeddings.pkl')

In [6]:
df_with_embeddings

,from_year,to_year,year_N,year_N1,HS_N,HS_N1,Cosine_Similarity,Jaccard_Similarity,Levenshtein_Similarity,Combined_Similarity,...,embedding_similarity,best_match_hscode,best_match_similarity,best_match_description,best_match_source,query_cosine_similarity,query_jaccard_similarity,query_levenshtein_similarity,query_combined_similarity,description_embedding
0,2022,2023,2022,2023,01023100,01023100,1.000000,1.000000,1.000000,1.000000,...,1.000000,01023100,1.0,Live purebred breeding buffalo,2023_mapping,0.0,0.0,0.0,0.0,"[-0.09349966049194336, 0.215228870511055, 0.03..."
1,2022,2023,2022,2023,01019040,01019040,1.000000,1.000000,1.000000,1.000000,...,1.000000,01019040,1.0,Mules and hinnies not imported for immediate s...,2023_mapping,0.0,0.0,0.0,0.0,"[0.03550305590033531, 0.245530903339386, 0.074..."
2,2022,2023,2022,2023,01031000,01031000,1.000000,1.000000,1.000000,1.000000,...,1.000000,01031000,1.0,Live purebred breeding swine,2023_mapping,0.0,0.0,0.0,0.0,"[0.006545722484588623, 0.09559973329305649, 0...."
3,2022,2023,2022,2023,01022100,01022100,1.000000,1.000000,1.000000,1.000000,...,1.000000,01022100,1.0,Live purebred breeding cattle,2023_mapping,0.0,0.0,0.0,0.0,"[-0.05505869910120964, 0.07829249650239944, 0...."
4,2022,2023,2022,2023,01012900,01012900,1.000000,1.000000,1.000000,1.000000,...,1.000000,01012900,1.0,Live horses other than purebred breeding horses,2023_mapping,0.0,0.0,0.0,0.0,"[0.07840116322040558, 0.20285926759243011, -0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
745501,1789,1790,1789,1790,1789_56_canes_walking_sticks_and_whips,1790_129_canes_walking_sticks_and_whips,1.000000,1.000000,1.000000,1.000000,...,0.094847,6602,0.827552,"Walking-sticks, seat-sticks, whips, riding-cro...",HS4,0.0,0.0,0.0,0.0,"[0.24178597331047058, -0.12082578986883163, 0...."
745502,1789,1790,1789,1790,1789_61_playing_cards,1790_110_playing_cards,1.000000,1.000000,1.000000,1.000000,...,0.035573,950440,0.893616,Games; playing cards,HS6,0.0,0.0,0.0,0.0,"[0.020294873043894768, -0.14760056138038635, -..."
745503,1789,1790,1789,1790,1789_62_every_coach_chariot_or_other_four_wheel,1790_138_all_coaches_chariots_phaetons_chaises_ch,0.836723,0.120000,0.594340,0.669140,...,0.206852,940191,0.604655,"Seats; parts, of wood, (other than for use in ...",HS6,0.0,0.0,0.0,0.0,"[0.073908731341362, 0.02879342809319496, 0.034..."
745504,1789,1790,1789,1790,1789_58_brushes,1790_131_brushes,1.000000,1.000000,1.000000,1.000000,...,0.215003,960330,0.754444,"Brushes; artists' brushes, writing brushes and...",HS6,0.0,0.0,0.0,0.0,"[0.10028719902038574, -0.1482127457857132, -0...."


In [ ]:
# Later, when you need to search:
query = "Cotton cloth, bleached, containing synthetic fiber"
similar_items = find_similar_descriptions(
    query, 
    df_with_embeddings, 
    embedding_column='description_embedding',
    description_column='Description_N', 
    top_k=5
)

# Display results
for idx, row in similar_items.iterrows():
    print(f"Similarity: {row['query_combined_similarity']:.3f}")
    print(f"Description: {row['Description_N']}")
    print(f"2023 HS Code: {row['best_match_hscode']}")
    print(f"2023 Description: {row['best_match_description']}")
    print("-" * 50)

Similarity: 0.745
Description: Cotton cloth, bleached
HS Code: 1894_2951_cotton_cloth_bleached
--------------------------------------------------
Similarity: 0.685
Description: Cotton cloth not bleached, dyed, colored, stained, painted, or cloth printed
HS Code: 1894_2939_cotton_cloth_not_bleached_dyed_colored_s
--------------------------------------------------
Similarity: 0.668
Description: Cotton cloths
HS Code: 1828_500_cotton_cloths
--------------------------------------------------
Similarity: 0.639
Description: Bleached plain weave fabrics of cotton, under 85% cotton wt., mixed mainly with man-made fibers, n/o 200 g/m2, of number 42 or lower
HS Code: 52102140
--------------------------------------------------
Similarity: 0.639
Description: Bleached plain weave fabrics of cotton, under 85% cotton wt., mixed mainly with man-made fibers, n/o 200 g/m2, of number 42 or lower
HS Code: 52102140
--------------------------------------------------
